Dataset Link: https://www.kaggle.com/datasets/pythonafroz/state-of-charge-and-state-of-health-of-batteries/data

In [1]:
import warnings 
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
base_path = "."
use_aged = False   # Set True for aged cell

if use_aged:
    exp_file = f"{base_path}/Dataset/Experimental_data_aged_cell.csv"
else:
    exp_file = f"{base_path}/Dataset/Experimental_data_fresh_cell.csv"

ocv_file = f"{base_path}/Dataset/OCV_vs_SOC_curve.csv"

df = pd.read_csv(exp_file)
df_ocv = pd.read_csv(ocv_file)

# Standardize column names
df.columns = [c.strip().title() for c in df.columns]
df.head()

,Time,Current,Voltage,Temperature
0,0.000000,2.158704,2.999607,26.384377
1,1.000000,2.287674,2.999407,26.227879
2,2.000000,2.228280,2.999757,26.449251
3,2.999992,2.224886,2.999857,26.277494
4,4.000000,2.134946,2.999958,26.380539


# SOC Estimation (Coulomb Counting)

In [4]:
df = df.copy()
dt = df['Time'].diff().fillna(0.0)

soc = np.zeros(len(df))
soc[0] = 0.5
capacity_Ah = 19.96
cap_As = capacity_Ah * 3600.0

for i in range(1, len(df)):
    soc[i] = soc[i-1] - (dt.iloc[i] / cap_As) * df['Current'].iloc[i]
    if (df['Voltage'].iloc[i] > 4.19 and abs(df['Current'].iloc[i]) < 4) or soc[i] > 1:
        soc[i] = 1.0
    if (df['Voltage'].iloc[i] < 3.01 and abs(df['Current'].iloc[i]) < 4) or soc[i] < 0:
        soc[i] = 0.0

df['SOC_CC'] = soc


# Reference SOH Estimation

In [5]:
# df = df.copy()
# capacity_Ah_nom = 19.96
# cap_As_nom = capacity_Ah_nom * 3600.0
# window_As_mult = 0.05

# C_exp = 0.0
# soh_points, t_points = [], []

# for i in range(1, len(df)):
#     dt = df['Time'].iloc[i] - df['Time'].iloc[i-1]
#     C_exp += abs(df['Current'].iloc[i]) * dt
#     if C_exp > window_As_mult * cap_As_nom:
#         soh_points.append(C_exp / (window_As_mult * cap_As_nom))
#         t_points.append(df['Time'].iloc[i])
#         C_exp = 0.0

# if len(t_points) >= 2:
#     df['SOH_ref'] = np.interp(df['Time'], t_points, soh_points)
# else:
#     df['SOH_ref'] = np.nan


In [6]:
df = df.copy()
capacity_Ah_nom = 19.96
cap_As_nom = capacity_Ah_nom * 3600.0

# crude cycle detection: whenever voltage falls below ~3.1V we assume a discharge cycle
df['Cycle'] = (df['Voltage'] < 3.1).cumsum()

soh_values = {}
for cycle_id, group in df.groupby('Cycle'):
    if len(group) > 10:  # skip tiny segments
        delivered_As = np.trapz(group['Current'], group['Time'])  # integrate current over time
        soh = abs(delivered_As) / cap_As_nom
        soh_values[cycle_id] = soh

df['SOH_ref'] = df['Cycle'].map(soh_values)

# Feature Engineering

In [7]:
df = df.copy()
df['dVdt'] = df['Voltage'].diff().fillna(0.0) / df['Time'].diff().replace(0, np.nan).fillna(1.0)
df['dIdt'] = df['Current'].diff().fillna(0.0) / df['Time'].diff().replace(0, np.nan).fillna(1.0)
df['absI'] = df['Current'].abs()
df['I2'] = df['Current']**2
df['TempC'] = df['Temperature']

# OCV Lookup
soc_vals = df_ocv['SOC'].values
v0_vals = df_ocv['V0'].values
df['V0_hat'] = np.interp(np.clip(df['SOC_CC'].values, 0, 1), soc_vals, v0_vals)

# Internal resistance proxy
df['R_hat'] = (df['V0_hat'] - df['Voltage']).replace([np.inf, -np.inf], np.nan)

# Rolling stats
for col in ['Current','Voltage','Temperature','dVdt','dIdt','R_hat']:
    df[f'{col}_mean_60s'] = df[col].rolling(60, min_periods=1).mean()
    df[f'{col}_std_60s'] = df[col].rolling(60, min_periods=1).std().fillna(0.0)

df['SOC_target'] = df['SOC_CC']


In [8]:
df = df.dropna(subset=['SOH_ref']).reset_index(drop=True)

feature_cols = [c for c in df.columns if any(k in c for k in [
    'Current','Voltage','Temperature','dVdt','dIdt','absI','I2','V0_hat','R_hat','mean_60s','std_60s'])]

In [9]:
dataset = df[feature_cols + ['SOC_target', 'SOH_ref']]

# Save to CSV for later use
dataset.to_csv("processed_dataset.csv", index=False)

In [10]:
dataset.head()

,Current,Voltage,Temperature,dVdt,dIdt,absI,I2,V0_hat,R_hat,Current_mean_60s,...,Temperature_mean_60s,Temperature_std_60s,dVdt_mean_60s,dVdt_std_60s,dIdt_mean_60s,dIdt_std_60s,R_hat_mean_60s,R_hat_std_60s,SOC_target,SOH_ref
0,2.067067,2.999707,26.262211,0.000451,-0.045819,2.067067,4.272768,3.010470,0.010763,2.140559,...,26.304795,0.092747,0.000008,0.000266,-0.007049,0.054307,0.069869,0.212613,0.000000,0.02211
1,-19.932451,3.100338,26.178240,0.100630,-21.999351,19.932451,397.302603,3.016418,-0.083921,0.563916,...,26.295755,0.095312,0.007195,0.026894,-1.577928,5.877921,0.058884,0.208366,0.000277,0.02211
2,-20.000328,3.112109,26.266050,0.011771,-0.067877,20.000328,400.013120,3.022385,-0.089724,-0.807034,...,26.293775,0.092165,0.007500,0.025942,-1.477258,5.677510,0.048977,0.204420,0.000556,0.02211
3,-19.981662,3.121376,26.372910,0.009267,0.018666,19.981662,399.266816,3.028347,-0.093029,-2.005448,...,26.298721,0.091211,0.007611,0.025067,-1.383762,5.497731,0.040101,0.200654,0.000834,0.02211
4,-19.920572,3.129440,26.285124,0.008064,0.061090,19.920572,396.829189,3.033872,-0.095569,-3.059279,...,26.297921,0.088376,0.007637,0.024271,-1.298771,5.334677,0.032121,0.197049,0.001111,0.02211


In [11]:
null_counts = dataset.isnull().sum()
# Print the number of null values
print(f"{null_counts.sum()} null entries have been found in the datasetset\n")
# Drop null values
dataset.dropna(inplace=True)          # or dataset_dataset = dataset_dataset.dropna()

# Find and handle duplicates
duplicate_count = dataset.duplicated().sum()
# Print the number of duplicate entries
print(f"{duplicate_count} duplicate entries have been found in the datasetset\n")
# Remove duplicates
dataset.drop_duplicates(inplace=True)  # or dataset_dataset = dataset_dataset.drop_duplicates()
# Display relative message
print(f"All duplicates have been removed\n")

# Reset the indexes
dataset.reset_index(drop=True, inplace=True)

# Inspect the datasetset for categorical columns
print("Categorical columns:",dataset.select_dtypes(include=['object']).columns.tolist(),'\n')

0 null entries have been found in the datasetset

0 duplicate entries have been found in the datasetset

All duplicates have been removed

Categorical columns: [] 



In [12]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353369 entries, 0 to 353368
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Current               353369 non-null  float64
 1   Voltage               353369 non-null  float64
 2   Temperature           353369 non-null  float64
 3   dVdt                  353369 non-null  float64
 4   dIdt                  353369 non-null  float64
 5   absI                  353369 non-null  float64
 6   I2                    353369 non-null  float64
 7   V0_hat                353369 non-null  float64
 8   R_hat                 353369 non-null  float64
 9   Current_mean_60s      353369 non-null  float64
 10  Current_std_60s       353369 non-null  float64
 11  Voltage_mean_60s      353369 non-null  float64
 12  Voltage_std_60s       353369 non-null  float64
 13  Temperature_mean_60s  353369 non-null  float64
 14  Temperature_std_60s   353369 non-null  float64
 15  

In [13]:
import joblib
joblib.dump(feature_cols, "Models/feature_columns.pkl")

['Models/feature_columns.pkl']